# M3 Keycloak JWT demo

Notebook didattico per mostrare il flow completo:

1. Keycloak emette un access token con `client_credentials`.
2. Il client usa il token per chiamare l'API FastAPI protetta.
3. Il backend verifica JWT, issuer e client atteso.

Questo notebook assume che l'esempio sia avviato localmente.

## Parametri di connessione

Se stai usando la configurazione standard del repository, i valori di default sono:

- Keycloak: `http://localhost:8080`
- API protetta: `http://localhost:8001`
- Realm: `m3-jwt`
- Client id: `m3-fastapi-client`
- Client secret: `m3-fastapi-secret`

In [ ]:
import os
import requests

KEYCLOAK_BASE_URL = os.getenv('KEYCLOAK_BASE_URL', 'http://localhost:8080')
API_BASE_URL = os.getenv('API_BASE_URL', 'http://localhost:8001')
REALM = os.getenv('REALM', 'm3-jwt')
CLIENT_ID = os.getenv('CLIENT_ID', 'm3-fastapi-client')
CLIENT_SECRET = os.getenv('CLIENT_SECRET', 'm3-fastapi-secret')

TOKEN_URL = f'{KEYCLOAK_BASE_URL}/realms/{REALM}/protocol/openid-connect/token'
PROTECTED_URL = f'{API_BASE_URL}/protected'

TOKEN_URL, PROTECTED_URL

In [ ]:
token_response = requests.post(
    TOKEN_URL,
    data={
        'grant_type': 'client_credentials',
        'client_id': CLIENT_ID,
        'client_secret': CLIENT_SECRET,
    },
    timeout=30,
)
token_response.raise_for_status()
token_data = token_response.json()

access_token = token_data['access_token']
print('Token type:', token_data.get('token_type'))
print('Access token prefix:', access_token[:30] + '...')

In [ ]:
protected_response = requests.get(
    PROTECTED_URL,
    headers={'Authorization': f'Bearer {access_token}'},
    timeout=30,
)
protected_response.raise_for_status()
protected_data = protected_response.json()
protected_data

## Da osservare

- il token viene ottenuto senza password utente, solo con client e segreto;
- l'API accetta richieste solo se il JWT e valido;
- il backend puo leggere `azp`, `iss` e `sub` dal token;
- il notebook riproduce lo stesso flusso del client Python e della collezione Postman.